# Inference using LSDB-style API

In [1]:
import lsdb

In [ ]:
from distributed import Client

N_WORKERS, THREADS_PER_WORKER = 1, 1
client = Client(n_workers=N_WORKERS, threads_per_worker=THREADS_PER_WORKER, processes=True)

## Open a HATS catalog with LSDB

Open a TESS catalog with LSDB and use LSDB's `map_partitions` to apply a function to each row in the catalog. In this example, we're applying a function that removes nested rows that contain NaNs.

In [2]:
from lsdb import ConeSearch

tess = lsdb.open_catalog(
    "https://data.lsdb.io/hats/tess/tess_lightcurve",
    columns=["ticid", "ra_obj", "dec_obj", "lightcurve"],
)

cone = ConeSearch(ra=180.0, dec=0.0, radius_arcsec=1 * 3600)
tess_cone = tess.search(cone)


def drop_nans(df):
    return df.dropna(subset="lightcurve.sap_flux").dropna(subset="lightcurve")


tess_filtered_nans = tess_cone.map_partitions(drop_nans)

In [ ]:
import pandas as pd
from dataclasses import dataclass

# Consider the two cases for config use of 1) A user has trained a model and is then using that model
# 2) A user is starting with a trained model and wants to run inference with it


@dataclass
class ModelConfig:
    model: str
    model_weights_file: str
    # batch_size: int = 64
    # hyrax_config: str = None


# this would just contain the most important confi parameters, if the user needs more
# control, then they would use the LSDB then Hyrax approach.
my_model_config = ModelConfig(
    model="HyraxLoopback",
    model_weights_file="./loopback_weights/fake_weights.pth",
    batch_size=64,
    # hyrax_config="/path/to/config"
)
my_model_config.use_config("/path/to/config")  # would call a bunch of .set_config(...)
my_model_config.set_field("data_loader.batch_size", 512)


PRED_META = pd.DataFrame(
    {
        "pred": pd.Series(dtype="float32"),
    }
)
met = tess_filtered_nans._operation.meta.copy(deep=True)
met["pred"] = PRED_META["pred"]
# met = met.join(PRED_META) # The more general approach

## Context manager approach

Here we don't necessarily need to create the client (and cluster?) before hand,
instead we rely on the `ml_client` context manager to handle client creation and
plugin registration automatically.

The only required parameter for the `ml_client` context manager is the hyrax
configuration in order to instantiate the model on the worker.

Optional kwargs are passed directly to the `Client` constructor inside the context
manager.

In [ ]:
from lsdb_ml.hyrax.client import ml_client
from lsdb_ml.hyrax.worker_plugin import predict

with ml_client(
    my_model_config,
    num_workers=4,
):  # + optional kwargs passed to Client(...)
    tess_pred = predict(tess_filtered_nans)  # the ideal setup
    tess_pred.compute()

    #     tess_filtered_nans,
    #     primary_id_field="ticid", # Move this to the model config
    #     input_fields=["ra_obj", "dec_obj", "lightcurve_sap_flux"], # Move this to the model config
    #     output_fields=PRED_META
    # )
    # tess_pred = tess_filtered_nans.map_partitions(
    #     predict,
    #     primary_id_field="ticid",  # optional, defaults to "_healpix_29"
    #     input_fields=["ra_obj", "dec_obj", "lightcurve_sap_flux"],  # optional, defaults to all fields
    #     output_fields=PRED_META,
    #     meta=met,
    # )
    tess_pred.compute()

http://127.0.0.1:8787/status


Computing Catalog:   0%|          | 0/4 [00:00<?, ?it/s]

2026-10-23 15:20:20,859 ignite.distributed.auto.auto_dataloader INFO: Use data loader kwargs for dataset '<torch.utils.data.da': 
	{'sampler': None, 'batch_size': 64, 'collate_fn': <bound method CollationMixin.collate of Name: data (primary dataset)
  Dataset class: DataframeDataset
  Data location: {'ticid': 'int64[pyarrow]', 'ra_obj': 'double[pyarrow]', 'dec_obj': 'double[pyarrow]', 'lightcurve': {'cadenceno': 'int32[pyarrow]', 'mom_centr1': 'double[pyarrow]', 'mom_centr1_err': 'float[pyarrow]', 'mom_centr2': 'double[pyarrow]', 'mom_centr2_err': 'float[pyarrow]', 'pdcsap_flux': 'float[pyarrow]', 'pdcsap_flux_err': 'float[pyarrow]', 'pos_corr1': 'float[pyarrow]', 'pos_corr2': 'float[pyarrow]', 'psf_centr1': 'double[pyarrow]', 'psf_centr1_err': 'float[pyarrow]', 'psf_centr2': 'double[pyarrow]', 'psf_centr2_err': 'float[pyarrow]', 'quality': 'int32[pyarrow]', 'sap_bkg': 'float[pyarrow]', 'sap_bkg_err': 'float[pyarrow]', 'sap_flux': 'float[pyarrow]', 'sap_flux_err': 'float[pyarrow]', 'ti

## Explicit client registration approach
This approach doesn't rely on a context manager and thus requires that the
Hyrax plugin be explicitly registered with the client before performing inference.
i.e. the user is responsible for creating the client and registering the plugin explicitly.

It also means that subsequent computation steps would be possible since the client
would not be closed out automatically.

In [ ]:
from lsdb_ml.hyrax.worker_plugin import predict, register_hyrax_plugin

register_hyrax_plugin(client, config)

tess_pred = tess_filtered_nans.map_partitions(
    predict,
    primary_id_field="ticid",
    input_fields=["ra_obj", "dec_obj", "lightcurve_sap_flux"],
    output_fields=PRED_META,
    meta=met,
)

# Potentially perform additional computation steps here...

tess_pred.compute()

In [ ]:
client.close()
cluster.close()